# Task 5 - Source metadata ingestion into MongoDB

Spark reads only `cpg.source-metadata.v1`, parses a fixed schema, and writes replacement upserts keyed by `_id=file_id`. The checkpoint volume is retained across restarts.

In [1]:
!docker compose exec -T mongo mongosh --quiet --eval "db=db.getSiblingDB('lab04'); printjson({documents:db.source_metadata.countDocuments({repo_id:'huggingface/optimum'}),distinct_files:db.source_metadata.distinct('_id',{repo_id:'huggingface/optimum'}).length}); printjson(db.source_metadata.findOne({_id:'867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7'}))"

{
  documents: 61,
  distinct_files: 61
}
{
  _id: '867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7',
  path: 'optimum/version.py',
  content_hash: '079eca803ea5bfe068bc805997b013cc14c9ddc8629a2ec634d12bcebb6720ce',
  node_counts: {
    AST: 25,
    SYNTHETIC: 4
  },
  edge_counts: {
    AST: 24,
    CFG: 9,
    DFG: 3
  },
  processed_at: '2026-07-20T06:02:53.464760Z',
  kafka_offset: Long('126')
}


In [1]:
!docker compose exec -T spark-metadata bash -lc "latest=$(find /opt/checkpoints/source-metadata-v1/offsets -maxdepth 1 -type f ! -name '.*' -printf '%f\n' | sort -n | tail -1); echo latest_batch=$latest; cat /opt/checkpoints/source-metadata-v1/offsets/$latest"

latest_batch=9
v1
{"batchWatermarkMs":0,"batchTimestampMs":1784527374985,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.optimizer.pruneFiltersCanPruneStreamingSubplan":"false","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.shuffle.partitions":"1"}}
{"cpg.source-metadata.v1":{"0":128}}


![MongoDB UI showing the upserted source metadata document](figures/mongodb-ui.png)

*MongoDB UI showing the upserted source metadata document*

## Reflection

A checkpoint stores Kafka progress, not file hashes. Unchanged files are skipped because their previous offsets remain committed and a single-file parser run does not re-emit other files.